# 📊 Chapter 7: Expected Value & Variance
*Tier 1 — Foundations | All Tracks*

> **🎬 Hook:** A casino game: roll a die, win $10 on 6, lose $2 otherwise. Should you play?
> Expected value says: play 1000 times and you'll lose money, guaranteed (in the long run).

**🎯 Objectives:** Compute E[X] and Var(X), understand what they represent, apply to decisions.

## 📖 Section 1 — Concept Review

### Expected Value E[X]
The **long-run average** value of a random variable over many trials.

$$E[X] = \sum_x x \cdot P(X=x) \quad \text{(discrete)}$$
$$E[X] = \int_{-\infty}^{\infty} x \cdot f(x) \, dx \quad \text{(continuous)}$$

### Variance Var(X)
How **spread out** the values are around the mean.
$$\text{Var}(X) = E[(X - \mu)^2] = E[X^2] - (E[X])^2$$

### Standard Deviation σ
$$\sigma = \sqrt{\text{Var}(X)}$$
Same units as X — much more interpretable than variance.

### Key Properties
- $E[aX + b] = aE[X] + b$ (linearity)
- $\text{Var}(aX + b) = a^2 \text{Var}(X)$
- $E[X + Y] = E[X] + E[Y]$ (always!)
- $\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y)$ only if independent

## 🧮 Section 2 — Math Walkthrough

### Casino Die Game
- Roll a fair die: win $10 on 6, lose $2 otherwise.

$$E[\text{profit}] = 10 \cdot P(6) + (-2) \cdot P(\text{not 6})$$
$$= 10 \cdot \frac{1}{6} + (-2) \cdot \frac{5}{6} = \frac{10 - 10}{6} = 0$$

This is a **fair game** — neither player has an advantage!

### Insurance Example
You pay $100/year for car insurance.
Without insurance: 1% chance of $8000 loss. Is insurance worth it?

- E[loss without insurance] = 0.01 × 8000 = **$80**
- Cost of insurance = **$100**
- E[loss with insurance] = $100 (certain)

Insurance costs more than expected loss — but reduces **variance** (uncertainty).
This is why risk-averse people buy insurance: they pay to reduce variance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")
np.random.seed(42)

# ── Casino Game Simulation ──
n_games = 50_000
rolls = np.random.randint(1, 7, n_games)
payoffs = np.where(rolls == 6, 10, -2)

running_avg = np.cumsum(payoffs) / np.arange(1, n_games + 1)
theoretical_ev = 10 * (1/6) + (-2) * (5/6)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(running_avg, color='#3498db', lw=1.5, alpha=0.9, label='Running average profit')
axes[0].axhline(theoretical_ev, color='#e74c3c', lw=2, linestyle='--',
                label=f'E[profit] = {theoretical_ev:.2f}')
axes[0].set_xlabel('Number of Games Played')
axes[0].set_ylabel('Average Profit per Game ($)')
axes[0].set_title(f'🎰 Casino Die Game: Convergence to E[X] = {theoretical_ev:.2f}', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(-5, 5)

# Compare high vs low variance investments
low_var  = np.random.normal(0.05, 0.05, 1000)   # bonds
high_var = np.random.normal(0.08, 0.25, 1000)   # stocks

axes[1].hist(low_var * 100, bins=40, alpha=0.6, color='#27ae60', density=True, label=f'Bonds: E={5:.0f}%, σ={5:.0f}%')
axes[1].hist(high_var * 100, bins=40, alpha=0.5, color='#e74c3c', density=True, label=f'Stocks: E={8:.0f}%, σ={25:.0f}%')
axes[1].axvline(5, color='#27ae60', lw=2, linestyle='--')
axes[1].axvline(8, color='#e74c3c', lw=2, linestyle='--')
axes[1].set_xlabel('Annual Return (%)')
axes[1].set_ylabel('Density')
axes[1].set_title('📈 Same(ish) E[X], Very Different Variance', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('ch07_ev_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"🎰 Casino game: E[profit] = {theoretical_ev:.2f} (simulated: {payoffs.mean():.3f})")
print(f"🎰 Casino game: Var[profit] = {payoffs.var():.2f}, Std = {payoffs.std():.2f}")

In [ ]:
# E[X] and Var(X) for common distributions
distributions = {
    'Bernoulli(p=0.3)':  {'dist': stats.bernoulli(0.3),  'theoretical': ('p=0.3', 'p(1-p)=0.21')},
    'Binomial(10,0.3)':  {'dist': stats.binom(10, 0.3),  'theoretical': ('np=3',  'np(1-p)=2.1')},
    'Poisson(λ=4)':      {'dist': stats.poisson(4),       'theoretical': ('λ=4',   'λ=4')},
    'Normal(5,2)':       {'dist': stats.norm(5, 2),       'theoretical': ('μ=5',   'σ²=4')},
    'Exponential(λ=1)':  {'dist': stats.expon(scale=1),   'theoretical': ('1/λ=1', '1/λ²=1')},
}

print(f"{'Distribution':<20} {'E[X] sim':>10} {'E[X] theory':>15} {'Var(X) sim':>12}")
print("-" * 60)
for name, info in distributions.items():
    samples = info['dist'].rvs(10_000)
    ev_sim = samples.mean()
    var_sim = samples.var()
    ev_th, var_th = info['theoretical']
    print(f"{name:<20} {ev_sim:>10.3f} {ev_th:>15} {var_sim:>12.3f}")

## 🔬 Section 3 — Simulation: Linearity of Expectation

In [ ]:
# Verify E[aX+b] = aE[X]+b and Var(aX+b) = a²Var(X)
np.random.seed(42)
X = np.random.normal(5, 2, 100_000)
a, b = 3, 10
Y = a * X + b

print("Linearity of Expectation & Variance:")
print(f"  E[X]     = {X.mean():.4f}  (theory: 5)")
print(f"  E[Y=3X+10] = {Y.mean():.4f}  (theory: {a*5+b})")
print()
print(f"  Var[X]   = {X.var():.4f}  (theory: 4)")
print(f"  Var[Y]   = {Y.var():.4f}  (theory: {a**2 * 4} = 3²×4)")
print()

# Sum of RVs
X1 = np.random.normal(3, 1, 100_000)
X2 = np.random.normal(7, 2, 100_000)
S  = X1 + X2
print(f"  E[X1+X2] = {S.mean():.4f}  (theory: {3+7}  = E[X1]+E[X2])")
print(f"  Var[X1+X2] = {S.var():.4f}  (theory: {1**2 + 2**2} = Var[X1]+Var[X2] since independent)")

## ✏️ Section 6 — Practice Problems

**P1:** You play a lottery: pay $2, win $100 with P=0.01, $0 otherwise. What is E[profit]? Should you play?
**P2:** X ~ Binomial(n=20, p=0.4). Find E[X] and Var(X).
**P3:** You have 2 investments: A has E=$1000, σ=$200; B has E=$1000, σ=$500. Which do you prefer and why?

<details><summary>💡 Solutions</summary>

**P1:** E[profit] = 100×0.01 + 0×0.99 − 2 = 1 − 2 = **−$1**. No, expected to lose $1 per play.

**P2:** E[X] = np = 20×0.4 = **8**, Var(X) = np(1-p) = 20×0.4×0.6 = **4.8**

**P3:** Same expected return → prefer A (lower variance = lower risk). Unless you need a big payoff, lower variance is better.
</details>

In [ ]:
# P1 solution
ev_lottery = 100 * 0.01 + 0 * 0.99 - 2
print(f"Lottery E[profit] = {ev_lottery}")
# P2 solution
print(f"Binomial(20,0.4): E={20*0.4}, Var={20*0.4*0.6}")

## 🎯 Recap
1. **E[X]** is the long-run average — it's where the distribution 'balances'.
2. **Var(X)** measures spread — high variance = more uncertainty.
3. **Linearity of expectation** always holds, even for dependent variables.

**Next:** [Chapter 8 — Common Distributions Part 1: Bernoulli, Binomial, Geometric]